# Face Mask Classification with YOLO Classification
This notebook assumes the notebook and `mask/` folder are in the same directory.

In [1]:
from pathlib import Path
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
import shutil

root=Path("mask")
data_dir=root/"data"
train_dir=root/"train"
val_dir=root/"val"

if not train_dir.exists():
    for cls in ["with_mask","without_mask"]:
        imgs=list((data_dir/cls).glob("*"))
        tr,va=train_test_split(imgs,test_size=0.2,random_state=42)
        for split,files in [("train",tr),("val",va)]:
            out=root/split/cls
            out.mkdir(parents=True,exist_ok=True)
            for f in files:
                dst=out/f.name
                if not dst.exists():
                    shutil.copy2(f,dst)
print("Dataset ready.")


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
model=YOLO("yolo11n-cls.pt")
model.train(
    data="mask",
    epochs=20,
    imgsz=224,
    batch=64,
    project="runs",
    name="mask_cls"
)


In [ ]:
best=YOLO("runs/classify/mask_cls/weights/best.pt")
best.export(format="onnx")
print("Exported ONNX")


In [ ]:
from sklearn.metrics import classification_report,confusion_matrix
import numpy as np

val_root=Path("mask/val")
true_labels=[]
pred_labels=[]

class_to_idx={"with_mask":0,"without_mask":1}

for cls in class_to_idx:
    for img in (val_root/cls).glob("*"):
        r=best.predict(str(img),verbose=False)[0]
        pred=int(r.probs.top1)
        pred_labels.append(pred)
        true_labels.append(class_to_idx[cls])

print(classification_report(true_labels,pred_labels,target_names=["with_mask","without_mask"]))
print(confusion_matrix(true_labels,pred_labels))
print("Accuracy:",np.mean(np.array(true_labels)==np.array(pred_labels)))


In [ ]:
from ultralytics import YOLO
import time
import pandas as pd
onnx_model=YOLO("runs/classify/mask_cls/weights/best.onnx",task="classify")

eval_dir=Path("mask/eval")
rows=[]
times=[]
for img in sorted(eval_dir.glob("*")):
    t=time.time()
    r=onnx_model.predict(str(img),verbose=False)[0]
    times.append(time.time()-t)
    rows.append({"image":img.name,
                 "prediction":"with_mask" if int(r.probs.top1)==0 else "without_mask"})
print(f"Average inference: {np.mean(times)*1000:.2f} ms")
print(f"FPS: {1/np.mean(times):.2f}")
submission=pd.DataFrame(rows)
submission.to_csv("submission.csv",index=False)
submission.head()
